# Formative 2, Task 6: System Simulation

**Author:** Bakhit. This notebook demonstrates the full multimodal authentication and product recommendation flow, including authorized and unauthorized attempts.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), ".."))

from scripts.predict_face import predict_face
from scripts.predict_product import predict_product
from scripts.verify_voice import verify_voice

print("all three models loaded successfully")

all three models loaded successfully


## 1. System Flow Overview

The system flow matches the assignment diagram: a user presents a face image to the face recognition gate first, since identity has to be confirmed before anything else runs. If the face is recognized, the system runs the product recommendation model for that user, then asks for a voice sample as a second, independent identity check. Only if voice verification also passes does the system display the recommended product. If either gate fails, access is denied immediately and the flow stops, so a stranger can never reach the product model or the final display just by getting past one check.

In [2]:
def simulate_transaction(face_path, voice_path, label=""):
    print("=" * 60)
    print(label)
    print("=" * 60)

    print("\nSTEP 1: Face recognition gate")
    face_label, face_confidence = predict_face(face_path)
    print(f"  input: {face_path}")
    print(f"  result: {face_label}, confidence={face_confidence:.3f}")
    if face_label == "unauthorized":
        print("\nACCESS DENIED: face not recognized")
        return {
            "Simulation": label, "Face Input": face_path,
            "Face Result": f"{face_label} ({face_confidence:.3f})",
            "Voice Input": "-", "Voice Result": "-",
            "Product Shown": "-", "Outcome": "DENIED (face)"
        }

    print("\nSTEP 2: Product recommendation")
    sample_features = {
        "engagement_score": 75.0,
        "purchase_interest_score": 3.5,
        "sentiment_score": 0.5,
        "n_platforms": 2,
        "n_transactions": 10,
        "total_spend": 500.0,
        "avg_amount": 50.0,
        "max_amount": 150.0,
        "avg_rating": 4.0,
        "n_categories": 3,
        "spend_per_txn": 50.0,
        "interest_x_engagement": 262.5,
        "is_high_value": 1,
    }
    product = predict_product(sample_features)
    print(f"  recommended product: {product}")

    print("\nSTEP 3: Voice verification gate")
    voice_label, voice_confidence = verify_voice(voice_path)
    print(f"  input: {voice_path}")
    print(f"  result: {voice_label}, confidence={voice_confidence:.3f}")
    if voice_label == "unauthorized":
        print("\nACCESS DENIED: voice not verified")
        return {
            "Simulation": label, "Face Input": face_path,
            "Face Result": f"{face_label} ({face_confidence:.3f})",
            "Voice Input": voice_path,
            "Voice Result": f"{voice_label} ({voice_confidence:.3f})",
            "Product Shown": product, "Outcome": "DENIED (voice)"
        }

    print("\n" + "=" * 60)
    print("TRANSACTION APPROVED")
    print(f"  user: {face_label}")
    print(f"  recommended product: {product}")
    print("=" * 60)
    return {
        "Simulation": label, "Face Input": face_path,
        "Face Result": f"{face_label} ({face_confidence:.3f})",
        "Voice Input": voice_path,
        "Voice Result": f"{voice_label} ({voice_confidence:.3f})",
        "Product Shown": product, "Outcome": "APPROVED"
    }

## 2. Simulation 1: Authorized Transaction

Bakhit presents his own face photo and voice clip. Both should pass the authentication gates, and the system should display a product recommendation. This proves the full end-to-end flow works for a legitimate user.

In [3]:
result_1 = simulate_transaction(
    face_path="../images/Bakhit/neutral.jpg",
    voice_path="../audio/Bakhit2.wav",
    label="SIMULATION 1: Authorized Transaction (Bakhit)"
)

SIMULATION 1: Authorized Transaction (Bakhit)

STEP 1: Face recognition gate


  input: ../images/Bakhit/neutral.jpg
  result: Bakhit, confidence=0.925

STEP 2: Product recommendation
  recommended product: Electronics

STEP 3: Voice verification gate


  input: ../audio/Bakhit2.wav
  result: Bakhit, confidence=0.960

TRANSACTION APPROVED
  user: Bakhit
  recommended product: Electronics


Bakhit's face passes the gate at 0.925 confidence, well above the 0.55 threshold, and the product model recommends Electronics. His voice clip then passes the second gate at 0.960 confidence, above the 0.6 threshold, so the transaction is approved end to end with Electronics as the recommended product.

## 3. Simulation 2: Unauthorized Face Attempt

A stranger's photo is presented instead of a team member's. The face gate should reject it immediately, at 0.325 confidence, well under the 0.55 threshold, and the system should never reach the product model or the voice gate. This proves the first security checkpoint works on its own.

In [4]:
result_2 = simulate_transaction(
    face_path="../images/unauthorized/attempt1.jpg",
    voice_path="../audio/Bakhit2.wav",
    label="SIMULATION 2: Unauthorized Face Attempt"
)

SIMULATION 2: Unauthorized Face Attempt

STEP 1: Face recognition gate
  input: ../images/unauthorized/attempt1.jpg
  result: unauthorized, confidence=0.325

ACCESS DENIED: face not recognized


The stranger's photo is rejected at 0.325 confidence, well under the 0.55 threshold. Access is denied at step 1, and the output above shows the flow never reaches step 2 or step 3, so the product model and the voice gate never run for this attempt.

## 4. Simulation 3: Unauthorized Voice Attempt

Bakhit's own photo passes the face gate, so the flow reaches the product model and the voice gate, but a stranger's voice clip is presented instead of Bakhit's own. The voice gate should reject that clip at 0.340 confidence, under the 0.6 threshold. This proves that passing the face gate alone is not enough: both biometric checks have to succeed before the product is actually shown.

In [5]:
result_3 = simulate_transaction(
    face_path="../images/Bakhit/neutral.jpg",
    voice_path="../audio/stranger1.wav",
    label="SIMULATION 3: Unauthorized Voice Attempt"
)

SIMULATION 3: Unauthorized Voice Attempt

STEP 1: Face recognition gate
  input: ../images/Bakhit/neutral.jpg
  result: Bakhit, confidence=0.925

STEP 2: Product recommendation
  recommended product: Electronics

STEP 3: Voice verification gate


  input: ../audio/stranger1.wav
  result: unauthorized, confidence=0.340

ACCESS DENIED: voice not verified


Bakhit's face passes at 0.925 confidence and the product model still runs, recommending Electronics, since the face gate alone does not block anything. The stranger1 voice clip is then rejected at 0.340 confidence, well under the 0.6 threshold, so the system correctly denies access at step 3 instead of approving the transaction on a passed face check alone.

## 5. Simulation 4: Second Unauthorized Voice Attempt

This repeats simulation 3 with the second stranger clip instead of the first, since stranger2.wav is rejected at 0.365 confidence in prior testing, a different number from stranger1's 0.340. Running both shows the voice gate consistently rejects unknown voices, not just one specific recording that happens to sit far below the threshold.

In [6]:
result_4 = simulate_transaction(
    face_path="../images/Bakhit/neutral.jpg",
    voice_path="../audio/stranger2.wav",
    label="SIMULATION 4: Second Unauthorized Voice Attempt"
)

SIMULATION 4: Second Unauthorized Voice Attempt

STEP 1: Face recognition gate
  input: ../images/Bakhit/neutral.jpg
  result: Bakhit, confidence=0.925

STEP 2: Product recommendation
  recommended product: Electronics

STEP 3: Voice verification gate
  input: ../audio/stranger2.wav
  result: unauthorized, confidence=0.365

ACCESS DENIED: voice not verified


stranger2's voice is rejected at 0.365 confidence, close to but still higher than stranger1's 0.340 confidence from simulation 3. Both numbers sit well below the 0.6 threshold, so the voice gate rejects both stranger recordings even though they are not the same distance from the cutoff.

## 6. Summary of All Simulations

In [7]:
import pandas as pd

summary_df = pd.DataFrame([result_1, result_2, result_3, result_4],
                          columns=["Simulation", "Face Input", "Face Result",
                                   "Voice Input", "Voice Result", "Product Shown", "Outcome"])
display(summary_df)

,Simulation,Face Input,Face Result,Voice Input,Voice Result,Product Shown,Outcome
0,SIMULATION 1: Authorized Transaction (Bakhit),../images/Bakhit/neutral.jpg,Bakhit (0.925),../audio/Bakhit2.wav,Bakhit (0.960),Electronics,APPROVED
1,SIMULATION 2: Unauthorized Face Attempt,../images/unauthorized/attempt1.jpg,unauthorized (0.325),-,-,-,DENIED (face)
2,SIMULATION 3: Unauthorized Voice Attempt,../images/Bakhit/neutral.jpg,Bakhit (0.925),../audio/stranger1.wav,unauthorized (0.340),Electronics,DENIED (voice)
3,SIMULATION 4: Second Unauthorized Voice Attempt,../images/Bakhit/neutral.jpg,Bakhit (0.925),../audio/stranger2.wav,unauthorized (0.365),Electronics,DENIED (voice)


The summary table shows the authorized transaction approved with Electronics recommended, and both unauthorized attempts correctly blocked: simulation 2 stopped at the face gate (0.325 confidence) before reaching the product model, while simulations 3 and 4 passed the face gate (0.925 confidence, same photo each time) but were stopped at the voice gate (0.340 and 0.365 confidence). A face-only system would have approved simulations 3 and 4 as soon as Bakhit's own photo passed, never checking that the voice belonged to a stranger. A voice-only system would never have caught the face stranger in simulation 2, since it would skip identity checking on the photo entirely and only ever ask for a voice sample.

## 7. System Limitations

- The face model reaches 1.000 accuracy on 4 test photos, but 4 photos is a very small test set, and it may not generalize to photos taken in different lighting or camera angles than the training set.
- The voice model reaches 0.500 accuracy on 4 test clips, with David and Serein consistently misclassified. A production system would need more training clips per member and longer recordings.
- This simulation uses hardcoded sample features for the product model instead of real data from the merged customer dataset. In a real deployment, those features would come from the authenticated user's actual social media profile and transaction history.
- Bakhit is the demo user here because he is the only member reliably recognized by both the face model and the voice model. A production system would need that same reliability across every enrolled member.
- The confidence thresholds used here, 0.55 for face and 0.6 for voice, were tuned to this specific dataset. A different user base or recording environment would need its own tuning.

## References

This notebook uses models trained in the following companion notebooks:

- Task 1: `notebooks/task1_data_merge.ipynb` (product recommendation model, Serein)
- Task 2: `notebooks/B_image_face_model.ipynb` (facial recognition model, David)
- Task 3: `notebooks/C_audio_voice_model.ipynb` (voiceprint verification model, Divine)